# Data Preparation and Generation

This notebook selects DC charging-station records from the cleaned Assignment 1 dataset, prepares the generator input, and uses the supplied generator to simulate hourly prices, charging sessions, and meter readings. The generation targets are at least 10,000 hourly prices and approximately 4,000 charging sessions.

In [ ]:
import csv
import hashlib
import json
import math
import re
import subprocess
import sys
from collections import Counter, defaultdict
from datetime import datetime, timedelta
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/processed/chargers_clean.csv").is_file() and (p / "assignment2/generator/generate_dataset.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from within the project.")
SOURCE = ROOT / "data/processed/chargers_clean.csv"
INPUT = ROOT / "assignment2/data/input/stations.csv"
OUTPUT = ROOT / "assignment2/data/generated/dc_data"
GENERATOR = ROOT / "assignment2/generator/generate_dataset.py"
START_DATE = "2026-09-15"
DAYS = 1
SEED = 5339
SESSION_RATES = [3.1, 2.1]

def read_csv(path):
    with path.open(newline="", encoding="utf-8-sig") as stream:
        return list(csv.DictReader(stream))

def write_csv(path, rows, fields):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

source_rows = read_csv(SOURCE)
dc_rows = [r for r in source_rows if r["charger_type_standardized"] == "DC"]
if not dc_rows or len({r["record_id"] for r in dc_rows}) != len(dc_rows):
    raise ValueError("Missing DC records or duplicate source identifiers.")
print(f"A1 rows: {len(source_rows)}; DC rows: {len(dc_rows)}")
print(f"Expected hourly prices: {len(dc_rows) * DAYS * 24}")

A1 rows: 1958; DC rows: 433
Expected hourly prices: 10392


## 1. Fields and simulation assumptions

The generator input contains `station_id,name,lat,lon,site_type,connectors,max_kw`. Identifiers, coordinates, connector counts, and available power values come from A1. Missing power values are parsed from `Charger_rating`, using the lower explicitly stated kW rating when multiple values occur. Missing names combine operator and address; display names are limited to 90 characters, with unshortened names retained in the preparation records.

The urban/regional categories select two simulated demand models. Records in the explicit LGA list use urban; other records use regional. Six records without LGA metadata have documented address-based assignments. These are simulation conventions, not official urban/rural classifications.

In [ ]:
URBAN_LGAS = {
    "Bayside Council", "Blacktown City Council", "Blue Mountains City Council",
    "Burwood Council", "Camden Council", "Campbelltown City Council",
    "Canada Bay Council, City of", "City Of Canada Bay Council",
    "Canterbury Bankstown Council", "Cumberland Council", "Fairfield City Council",
    "Georges River Council", "Hawkesbury City Council",
    "Hornsby, The Council of the Shire of", "Hunters Hill, The Council of the Municipality of",
    "Inner West Council", "Ku-ring-gai Council", "Lane Cove Municipal Council",
    "Liverpool City Council", "Mosman Municipal Council", "North Sydney Council",
    "Northern Beaches Council", "Parramatta, Council of the City of", "Penrith City Council",
    "Randwick City Council", "Ryde, Council of the City of", "Strathfield",
    "Strathfield Municipal Council", "Sutherland Shire Council",
    "Sydney, Council of the City of", "The Hills Shire Council", "Waverley Council",
    "Willoughby City Council", "Wollondilly Shire Council", "Woollahra Municipal Council",
}
MISSING_LGA_RULES = {
    "ev_f4541d35e73670741678834a9099d01eafad51a684e29a77a94091c07ad8bec2": ("urban", "Source address: Kingsford"),
    "ev_1000c978d32395ddf6dbe8e4ed609f5700762989566e7e1c5ebcf182f04a4399": ("urban", "Source address: Balgowlah"),
    "ev_6c9d2b1726480ec136956e375e2a0b6a970532d7b6ad719f050946266568b917": ("regional", "Source address: Wollongong; regional simulation model"),
    "ev_9eeb85b4fe2cf21b3fee9c759e75a01177f8019fcb29d3068169ecdde1a18a2d": ("urban", "Source address: Campbelltown"),
    "ev_54c3831ece16cc05d25aba9a38dd71dab4176e7e10ba51add6eca709c448bd0d": ("urban", "Source address: Mount Annan"),
    "ev_8e508009b751691ba8fa15d29dbcbf4a0a0dc4e8d0992f0e3c0d9d5525b7b21c": ("regional", "Source address: Figtree; regional simulation model"),
}
KNOWN_COORDINATE_REVIEWS = {
    "ev_92a54038955adbf16c92062ab57a4b497be32cb889fcc2b6cb5d53a12c433e0c",
    "ev_7cde3d4e59cd084712a54481b1dd4295c6fd2f3ab426b9fa8c291f5893d98880",
}
FIELDS = ["station_id", "name", "lat", "lon", "site_type", "connectors", "max_kw"]
station_rows = []
audit_rows = []
for row in dc_rows:
    sid = row["record_id"]
    power_text = row["power_kw"].strip()
    if power_text:
        power = float(power_text)
        power_method = "A1 power_kw"
    else:
        rating = row["Charger_rating"].strip()
        matches = re.findall(r"(\d+(?:\.\d+)?)\s*kW", rating, flags=re.I)
        if matches:
            power = min(float(value) for value in matches)
        elif re.fullmatch(r"\d+(?:\.\d+)?", rating):
            power = float(rating)
        else:
            raise ValueError(f"Unresolved source rating: {sid}: {rating}")
        power_method = "Parsed original rating; minimum explicit kW rating"
    lga = row["LGANAME"].strip()
    if lga:
        site_type = "urban" if lga in URBAN_LGAS else "regional"
        site_method = "Explicit simulation LGA allowlist"
    else:
        if sid not in MISSING_LGA_RULES:
            raise ValueError(f"Unresolved site category: {sid}")
        site_type, site_method = MISSING_LGA_RULES[sid]
    lat, lon = float(row["Latitude"]), float(row["Longitude"])
    connectors = int(row["Number_of_plugs"])
    if not (all(math.isfinite(v) for v in [lat, lon, power]) and -90 <= lat <= 90 and -180 <= lon <= 180 and connectors > 0 and power > 0):
        raise ValueError(f"Invalid numeric input: {sid}")
    address = " ".join(row["Station_address"].split()).removesuffix(", Australia")
    name = row["Station_name"].strip() or f"{row['Operator'].strip()} - {address}"
    original_name = name
    if len(name) > 90:
        name = name[:87].rstrip() + "..."
    station_rows.append(dict(zip(FIELDS, [sid, name, row["Latitude"], row["Longitude"], site_type, connectors, power])))
    audit_rows.append({
        "station_id": sid, "source_address": row["Station_address"], "source_lga": lga,
        "source_rating": row["Charger_rating"], "original_power_kw": power_text,
        "max_kw": power, "power_method": power_method, "site_type": site_type,
        "site_method": site_method, "original_display_name": original_name,
        "name_shortened": name != original_name,
        "potential_duplicate_review": row["potential_duplicate_review"],
        "shared_coordinates": row["shared_coordinates"],
        "known_coordinate_review": sid in KNOWN_COORDINATE_REVIEWS,
    })
write_csv(INPUT, station_rows, FIELDS)
AUDIT = INPUT.with_name("stations_preparation.csv")
write_csv(AUDIT, audit_rows, list(audit_rows[0]))
print(f"Saved {len(station_rows)} stations")
print(f"Site categories: {dict(Counter(r['site_type'] for r in station_rows))}")
print(f"Connectors: {sum(r['connectors'] for r in station_rows)}")
print(f"Recovered power values: {sum(not r['original_power_kw'] for r in audit_rows)}")
print(f"Duplicate-review flags retained: {sum(r['potential_duplicate_review'] == 'True' for r in audit_rows)}")

Saved 433 stations
Site categories: {'urban': 245, 'regional': 188}
Connectors: 1658
Recovered power values: 6
Duplicate-review flags retained: 27


## 2. Generate simulated data

Use the supplied generator for a one-day simulation on 15 September 2026 with random seed 5339. Urban/regional session intensities are 3.1 and 2.1 per connector per day, targeting approximately 4,000 sessions. Actual counts depend on stochastic arrivals, price response, and connector availability.

In [ ]:
cache_name = "PRICE_AND_DEMAND_202609_NSW1.csv"
cache_target = OUTPUT / "aemo_cache" / cache_name
if not cache_target.is_file():
    raise FileNotFoundError("Place the AEMO NSW1 monthly CSV in the output aemo_cache directory.")
start = datetime.fromisoformat(START_DATE)
expected_hours = {start + timedelta(hours=h) for h in range(DAYS * 24)}
covered_hours = set()
for row in read_csv(cache_target):
    if row.get("REGION") != "NSW1":
        continue
    ts = datetime.strptime(row["SETTLEMENTDATE"], "%Y/%m/%d %H:%M:%S")
    hour = (ts - timedelta(minutes=5)).replace(minute=0, second=0)
    if hour in expected_hours and math.isfinite(float(row["RRP"])):
        covered_hours.add(hour)
if covered_hours != expected_hours:
    raise ValueError("AEMO hourly coverage is incomplete.")
command = [sys.executable, str(GENERATOR.relative_to(ROOT)), "--stations", str(INPUT.relative_to(ROOT)), "--start", START_DATE,
           "--days", str(DAYS), "--sessions-per-connector-day", *map(str, SESSION_RATES),
           "--seed", str(SEED), "--out", str(OUTPUT.relative_to(ROOT))]
result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode:
    raise RuntimeError(result.stderr or "Generation failed")

433 stations, 10392 hourly prices, 4027 charging-sessions, 123025 meter-events -> assignment2/data/generated/dc_data/
  urban    mean  32.6 min /  25.3 kWh   (ARENA 2023: 27.3 min / 27.0 kWh)
  regional mean  27.6 min /  25.0 kWh   (ARENA 2023: 25.9 min / 24.5 kWh)
  price $/kWh min 0.40  mean 0.67  max 0.92



## 3. Data validation

Check hourly price coverage, station/session identifiers, connector ranges, session timing, meter references, nondecreasing cumulative energy, and non-overlapping sessions per connector. Sessions ending after midnight are retained. Timestamp strings preserve the generator's timezone-naive format.

The volume checks require at least 10,000 hourly prices and use a project validation range of 3,900–4,400 sessions for the approximate 4,000-session target.

In [ ]:
datasets = {name: read_csv(OUTPUT / f"{name}.csv") for name in ["stations", "prices_hourly", "sessions", "session_metering"]}
stations, prices, sessions, meters = (datasets[n] for n in ["stations", "prices_hourly", "sessions", "session_metering"])
station_map = {r["station_id"]: r for r in station_rows}
session_map = {r["session_id"]: r for r in sessions}
expected_pairs = {(sid, hour.isoformat()) for sid in station_map for hour in expected_hours}
checks = {
    "all_dc_records_preserved": [r["station_id"] for r in stations] == [r["record_id"] for r in dc_rows],
    "hourly_price_coverage": len(prices) == len(expected_pairs) and {(r["station_id"], r["ts"]) for r in prices} == expected_pairs,
    "at_least_10000_prices": len(prices) >= 10000,
    "session_count_project_target": 3900 <= len(sessions) <= 4400,
    "unique_session_ids": len(session_map) == len(sessions) and bool(sessions),
    "valid_station_references": all(r["station_id"] in station_map for rows in [prices, sessions, meters] for r in rows),
    "valid_prices": all(math.isfinite(float(r['price_per_kwh'])) and float(r['price_per_kwh']) > 0 and math.isfinite(float(r['wholesale_per_kwh'])) for r in prices),
}
checks["session_times_and_connectors"] = all(
    start <= datetime.fromisoformat(r["start_ts"]) < start + timedelta(days=DAYS)
    and datetime.fromisoformat(r["start_ts"]) <= datetime.fromisoformat(r["charge_end_ts"]) <= datetime.fromisoformat(r["end_ts"])
    and 1 <= int(r["connector_id"]) <= station_map[r["station_id"]]["connectors"]
    for r in sessions)
by_connector = defaultdict(list)
for r in sessions:
    by_connector[(r["station_id"], r["connector_id"])].append(r)
checks["no_overlapping_sessions"] = True
for rows in by_connector.values():
    ordered = sorted(rows, key=lambda r: r["start_ts"])
    if any(a["end_ts"] > b["start_ts"] for a,b in zip(ordered, ordered[1:])):
        checks["no_overlapping_sessions"] = False
previous_energy = {}
meter_valid = bool(meters)
seen_meter_keys = set()
for r in meters:
    session = session_map.get(r["session_id"])
    key = (r["session_id"], r["ts"])
    energy = float(r["energy_kwh"])
    if (session is None or key in seen_meter_keys or not math.isfinite(energy) or energy < previous_energy.get(r["session_id"], 0)):
        meter_valid = False
        break
    if not (r["station_id"] == session["station_id"] and r["connector_id"] == session["connector_id"] and session["start_ts"] <= r["ts"] <= session["charge_end_ts"]):
        meter_valid = False
        break
    seen_meter_keys.add(key)
    previous_energy[r["session_id"]] = energy
checks["meter_references_times_and_energy"] = meter_valid and set(previous_energy) == set(session_map)
for name, rows in datasets.items():
    print(f"{name}: {len(rows)}")
for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
if not all(checks.values()):
    raise ValueError("Review failed checks before delivering this dataset.")

stations: 433
prices_hourly: 10392
sessions: 4027
session_metering: 123025
all_dc_records_preserved: PASS
hourly_price_coverage: PASS
at_least_10000_prices: PASS
session_count_project_target: PASS
unique_session_ids: PASS
valid_station_references: PASS
valid_prices: PASS
session_times_and_connectors: PASS
no_overlapping_sessions: PASS
meter_references_times_and_energy: PASS


## 4. Reproducibility records

Record data provenance, category rules, preparation assumptions, generation parameters, row counts, and validation results. SHA-256 hashes identify source data, generator, input, outputs, and AEMO cache for reproducibility. Per-record decisions are saved in `stations_preparation.csv`.

In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

notes = {
    "source": str(SOURCE.relative_to(ROOT)), "source_sha256": sha256(SOURCE),
    "selection": "All DC records in source order; no deduplication or renumbering",
    "urban_lga_allowlist": sorted(URBAN_LGAS), "missing_lga_rules": MISSING_LGA_RULES,
    "assumptions": [
        "Source records are simulation stations, not verified distinct physical sites.",
        "Existing A1 power values are retained; missing values use the minimum explicit original kW rating.",
        "Site categories are simulation conventions, not official geography.",
        "Plug counts are simulated independent connectors, not verified simultaneous physical capacity.",
        "Coordinates are preserved; known issues and duplicate-review flags remain in the audit.",
        "Display labels are limited to 90 characters; complete labels and addresses remain in the audit.",
        "No external geospatial verification was performed for the dataset.",
    ],
}
INPUT.with_name("stations_metadata.json").write_text(json.dumps(notes, indent=2) + "\n", encoding="utf-8")
report = {
    "start_date": START_DATE, "days": DAYS, "seed": SEED, "sessions_per_connector_day": SESSION_RATES,
    "source_sha256": sha256(SOURCE), "generator_sha256": sha256(GENERATOR),
    "station_count": len(station_rows), "connectors": sum(r["connectors"] for r in station_rows),
    "site_categories": dict(Counter(r["site_type"] for r in station_rows)),
    "counts": {name: len(rows) for name, rows in datasets.items()},
    "checks": checks,
    "last_session_end": max(r["end_ts"] for r in sessions),
    "input_sha256": sha256(INPUT),
    "output_sha256": {name: sha256(OUTPUT / f"{name}.csv") for name in datasets},
    "aemo_cache_sha256": sha256(cache_target),
    "scope": "Synthetic station, hourly price, charging session, and meter datasets for Assignment 2.",
}
(OUTPUT / "run_summary.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))

{
  "start_date": "2026-09-15",
  "days": 1,
  "seed": 5339,
  "sessions_per_connector_day": [
    3.1,
    2.1
  ],
  "source_sha256": "99c83df26c9f595dc42987832cb92b2e6a3ca98c8084f678de511c743d2060af",
  "generator_sha256": "5db5579ff135b8d137dde2b44d630747e8be2351b3560614a6444f88c6a39167",
  "station_count": 433,
  "connectors": 1658,
  "site_categories": {
    "urban": 245,
    "regional": 188
  },
  "counts": {
    "stations": 433,
    "prices_hourly": 10392,
    "sessions": 4027,
    "session_metering": 123025
  },
  "checks": {
    "all_dc_records_preserved": true,
    "hourly_price_coverage": true,
    "at_least_10000_prices": true,
    "session_count_project_target": true,
    "unique_session_ids": true,
    "valid_station_references": true,
    "valid_prices": true,
    "session_times_and_connectors": true,
    "no_overlapping_sessions": true,
    "meter_references_times_and_energy": true
  },
  "last_session_end": "2026-09-16T00:48:04",
  "input_sha256": "e1eebe24cde6666d7a1

## 5. Output files

 Paths below are relative to `assignment2`.

| File | Content and purpose |
|---|---|
| `data/input/stations.csv` |  Generator input |
| `data/input/stations_preparation.csv` |  Per-record preparation and review decisions |
| `data/input/stations_metadata.json` |  Provenance, classification rules, and assumptions |
| `data/generated/dc_data/stations.csv` |  Station attributes and base prices |
| `data/generated/dc_data/prices_hourly.csv` |  Hourly retail and wholesale prices in AUD/kWh |
| `data/generated/dc_data/sessions.csv` |  Session times, energy, and cost |
| `data/generated/dc_data/session_metering.csv` |  Voltage, current, power, cumulative energy, and battery state of charge |
| `data/generated/dc_data/run_summary.json` |  Generation settings, counts, checks, and file hashes |

Hourly prices support price-change events; sessions support start/end events; meter readings support charging telemetry events. `station_id` links to A1 records and `session_id` links sessions to readings. Meter `energy_kwh` is cumulative session energy and must not be summed across readings.